# BaseLoader 与 Document

`Document` 是加载结果的统一数据结构，`BaseLoader` 是文档加载器的官方基础接口。自定义 Loader 时，优先实现 `lazy_load()`，让调用方可以逐条消费文档而不必一次加载全部数据。

## 核心方法

- `lazy_load()`：返回 `Iterator[Document]`，适合大文件或大量数据。
- `load()`：BaseLoader 基于 `lazy_load()` 收集并返回 `list[Document]`。
- `alazy_load()` / `aload()`：异步版本。

Loader 只负责加载。应先调用 `load()`，再单独使用 TextSplitter 切分，不把切分策略硬编码进 Loader。

In [ ]:
from collections.abc import Iterator
from pathlib import Path

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document


class TextFileLoader(BaseLoader):
    """最小化的 TXT Loader。"""

    def __init__(self, path: str | Path, encoding: str = "utf-8") -> None:
        self.path = Path(path)
        self.encoding = encoding

    def lazy_load(self) -> Iterator[Document]:
        yield Document(
            page_content=self.path.read_text(encoding=self.encoding),
            metadata={
                "source": str(self.path),
                "encoding": self.encoding,
            },
        )


In [ ]:
CURRENT_DIR = Path.cwd().resolve()
RAG_DIR = next((path for path in (CURRENT_DIR, *CURRENT_DIR.parents) if path.name == "6-LangChain中的RAG"), CURRENT_DIR / "系统学习" / "6-LangChain中的RAG")
ASSET_DIR = RAG_DIR / "asset" / "load"
loader = TextFileLoader(ASSET_DIR / "01-langchain-utf-8.txt")

# 一次性加载为列表
documents = loader.load()
print(documents)

# 懒加载：逐条产生 Document
for document in loader.lazy_load():
    print(document.metadata, document.page_content[:100])


## 设计要点

1. `page_content` 只放后续需要语义检索的文本。
2. `metadata` 保留来源、页码、行号、时间和业务过滤字段。
3. Loader 处理读取、解析和必要校验；切分、Embedding 与存储属于后续阶段。
4. 面向大型数据源时优先使用 `lazy_load()`，避免一次占用过多内存。

## 官方资料

- [Document Loader integrations](https://docs.langchain.com/oss/python/integrations/document_loaders)
- [Document API](https://reference.langchain.com/python/langchain-core/documents/base/Document)
- [BaseLoader API](https://reference.langchain.com/python/langchain-core/document_loaders/base/BaseLoader)